In [144]:
import pandas as pd
import numpy as np
import time
from matplotlib import pyplot as plt
import seaborn as sns
from countryinfo import CountryInfo

In [145]:
train = pd.read_csv("../data/train/train.csv")

In [146]:
train.head()

,Unnamed: 0,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,year,month,day
0,479,1916-07-02,Chile,Uruguay,0.0,4.0,COPA,Buenos Aires,Argentina,True,1916,7,2
1,481,1916-07-06,Argentina,Chile,6.0,1.0,COPA,Buenos Aires,Argentina,False,1916,7,6
2,482,1916-07-08,Brazil,Chile,1.0,1.0,COPA,Buenos Aires,Argentina,True,1916,7,8
3,483,1916-07-10,Argentina,Brazil,1.0,1.0,COPA,Buenos Aires,Argentina,False,1916,7,10
4,485,1916-07-12,Brazil,Uruguay,1.0,2.0,COPA,Buenos Aires,Argentina,True,1916,7,12


In [147]:
train.drop(columns=['Unnamed: 0'], inplace=True)

In [148]:
train['result'] = np.select(
    [
        train['home_score'] > train['away_score'],
        train['home_score'] < train['away_score']
    ],
    [
        1,  
        2   
    ],
    default=0 
)

In [149]:
train.drop(columns=['home_score', 'away_score'], inplace=True)

## FIFA Rankings

In [150]:
fifa = pd.read_csv('../data/raw/fifa_ranking.csv')
min_date = fifa['rank_date'].min()
train = train[train['date'] >= min_date]

In [151]:

all_teams = pd.concat([train['home_team'], train['away_team']]).unique()

missing_teams = [team for team in all_teams if team not in fifa['country_full'].values]

print(f"Total unique teams in train: {len(all_teams)}")
print(f"Teams NOT found in fifa['country_full']: {len(missing_teams)}")
print(missing_teams)

Total unique teams in train: 147
Teams NOT found in fifa['country_full']: 11
['United States', 'Ivory Coast', 'DR Congo', 'South Korea', 'Czech Republic', 'Iran', 'China', 'North Korea', 'Cape Verde', 'Kyrgyzstan', 'Gambia']


In [152]:
fifa["country_full"] = fifa["country_full"].replace({
    "USA": "United States",
    "Côte d'Ivoire": "Ivory Coast",
    "Congo DR": "DR Congo",
    "Korea Republic": "South Korea",
    "Korea DPR": "North Korea",
    "Czechia": "Czech Republic",
    "IR Iran": "Iran",
    "China PR": "China",
    "Cabo Verde": "Cape Verde",
    "Kyrgyz Republic": "Kyrgyzstan",
    "The Gambia": "Gambia",
})

In [153]:
fifa.head()

,rank,country_full,country_abrv,total_points,previous_points,rank_change,confederation,rank_date
0,140.0,Brunei Darussalam,BRU,2.0,0.0,140,AFC,1992-12-31
1,33.0,Portugal,POR,38.0,0.0,33,UEFA,1992-12-31
2,32.0,Zambia,ZAM,38.0,0.0,32,CAF,1992-12-31
3,31.0,Greece,GRE,38.0,0.0,31,UEFA,1992-12-31
4,30.0,Algeria,ALG,39.0,0.0,30,CAF,1992-12-31


In [154]:
train = train.sort_index()

In [155]:
fifa["rank_date"] = pd.to_datetime(fifa["rank_date"])
train["date"] = pd.to_datetime(train["date"])

fifa = fifa.sort_values(["country_full", "rank_date"])
train = train.sort_values("date")

home = (
    fifa[["country_full", "rank_date", "total_points"]]
    .rename(columns={
        "country_full": "home_team",
        "rank_date": "date",
        "total_points": "home_points"
    })
    .sort_values(["home_team", "date"])
)

home["date"] = pd.to_datetime(home["date"])

home = home.sort_values(["date", "home_team"]).reset_index(drop=True)

train = train.copy()
train["date"] = pd.to_datetime(train["date"])
train = train.sort_values(["date", "home_team"]).reset_index(drop=True)

train = pd.merge_asof(
    train.sort_values("date"),
    home,
    on="date",
    by="home_team",
    direction="backward"
)

In [156]:
away = (
    fifa[["country_full", "rank_date", "total_points"]]
    .rename(columns={
        "country_full": "away_team",
        "rank_date": "date",
        "total_points": "away_points"
    })
    .sort_values(["away_team", "date"])
)

away["date"] = pd.to_datetime(away["date"])

away = away.sort_values(["date", "away_team"]).reset_index(drop=True)

train = train.copy()
train["date"] = pd.to_datetime(train["date"])
train = train.sort_values(["date", "away_team"]).reset_index(drop=True)

train = pd.merge_asof(
    train.sort_values("date"),
    away,
    on="date",
    by="away_team",
    direction="backward"
)

In [157]:
train.isna().sum()

date            0
home_team       0
away_team       0
tournament      0
city            0
country         0
neutral         0
year            0
month           0
day             0
result          0
home_points     7
away_points    13
dtype: int64

In [158]:
train.dropna(inplace=True)

In [159]:
train.isna().sum()

date           0
home_team      0
away_team      0
tournament     0
city           0
country        0
neutral        0
year           0
month          0
day            0
result         0
home_points    0
away_points    0
dtype: int64

## Location

In [160]:
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

geolocator = Nominatim(
    user_agent="nations-matches-prediction",
    timeout=10
)

geocode = RateLimiter(
    geolocator.geocode,
    min_delay_seconds=2,   # Nominatim requests at most ~1/sec
    max_retries=2,
    error_wait_seconds=5,
    swallow_exceptions=False
)

In [161]:
def get_capital(country):
    try:
        return CountryInfo(country).capital()
    except:
        return None

In [162]:
def get_log_lat(city):
    try:
        location = geolocator.geocode(city)
        print(location.latitude, location.longitude)
        return location.latitude, location.longitude
    except Exception as e:
        raise e
        return None

In [163]:
train['home_capital'] = train['home_team'].apply(get_capital)
train['away_capital'] = train['away_team'].apply(get_capital)

In [164]:
train['home_capital'] = train.apply(
    lambda row: 'London' if row['home_team'] == 'England' else row['home_capital'],
    axis=1
)

train['away_capital'] = train.apply(
    lambda row: 'London' if row['away_team'] == 'England' else row['away_capital'],
    axis=1
)

In [165]:
train['home_capital'] = train.apply(
    lambda row: "Cardiff" if row['home_team'] == 'Wales' else row['home_capital'],
    axis=1
)

train['away_capital'] = train.apply(
    lambda row: "Cardiff" if row['away_team'] == 'Wales' else row['away_capital'],
    axis=1
)

In [166]:
train['home_capital'] = train.apply(
    lambda row: "Belfast" if row['home_team'] == 'Northern Ireland' else row['home_capital'],
    axis=1
)

train['away_capital'] = train.apply(
    lambda row: "Belfast" if row['away_team'] == 'Northern Ireland' else row['away_capital'],
    axis=1
)

In [167]:
train['home_capital'] = train.apply(
    lambda row: "Pristina" if row['home_team'] == 'Kosovo' else row['home_capital'],
    axis=1
)

train['away_capital'] = train.apply(
    lambda row: "Pristina" if row['away_team'] == 'Kosovo' else row['away_capital'],
    axis=1
)

In [168]:
train.isna().sum()

date            0
home_team       0
away_team       0
tournament      0
city            0
country         0
neutral         0
year            0
month           0
day             0
result          0
home_points     0
away_points     0
home_capital    0
away_capital    0
dtype: int64

In [169]:
train['home_comb'] = train['home_capital'] + ", " + train['home_team']
train['away_comb'] = train['away_capital'] + ", " + train['away_team']
train['stadium_comb'] = train['city'] + ", " + train['country']

In [170]:
import time
import json
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderRateLimited

geolocator = Nominatim(
    user_agent="nations-matches-prediction",
    timeout=10
)

CACHE_FILE = "coords_cache.json"

# Load existing cache so you never re-geocode the same place
try:
    with open(CACHE_FILE) as f:
        coords = json.load(f)
except (FileNotFoundError, json.JSONDecodeError):
    coords = {}

def get_coords(place):
    if place in coords:
        return coords[place]  # skip already-geocoded places
    
    for attempt in range(5):
        try:
            time.sleep(1.2)  # always wait before each request
            location = geolocator.geocode(place)
            result = (location.latitude, location.longitude) if location else None
            coords[place] = result

            # Save after every successful call
            with open(CACHE_FILE, "w") as f:
                json.dump(coords, f)

            return result

        except GeocoderRateLimited:
            wait = 60
            print(f"Rate limited on '{place}', waiting {wait}s (attempt {attempt+1}/5)")
            time.sleep(wait)

    print(f"Giving up on '{place}' after 5 attempts")
    coords[place] = None
    return None

places = pd.concat([
    train['home_comb'],
    train['away_comb'],
    train['stadium_comb']
]).dropna().unique()

for place in places:
    result = get_coords(place)
    print(f"{place}: {result}")

Quito, Ecuador: [-0.2201641, -78.5123274]
Bogotá, Colombia: [4.6533817, -74.0836331]
Montevideo, Uruguay: [-34.9058916, -56.1913095]
Buenos Aires, Argentina: [-34.6095579, -58.3887904]
Asunción, Paraguay: [-25.2800459, -57.6343814]
Brasília, Brazil: [-15.7939869, -47.8828]
Washington D.C., United States: [38.8950982, -77.0363849]
Mexico City, Mexico: [19.3207722, -99.1514678]
Santiago, Chile: [-33.4376995, -70.6510671]
Lima, Peru: [-12.0459808, -77.0305912]
Abuja, Nigeria: [9.0643305, 7.4892974]
Tunis, Tunisia: [36.8002068, 10.1857757]
Accra, Ghana: [5.5571096, -0.2012376]
Yamoussoukro, Ivory Coast: [6.8200066, -5.2776034]
Cairo, Egypt: [30.0443879, 31.2357257]
Dakar, Senegal: [14.693425, -17.447938]
Lusaka, Zambia: [-15.4163395, 28.2818414]
Berlin, Germany: [52.5173885, 13.3951309]
Madrid, Spain: [40.416782, -3.703507]
Rome, Italy: [41.8933203, 12.4829321]
Yaoundé, Cameroon: [3.8689867, 11.5213344]
Oslo, Norway: [59.9133301, 10.7389701]
Brussels, Belgium: [50.8467372, 4.352493]
Amster

In [171]:
train[['home_lat', 'home_lon']] = (
    train['home_comb']
    .map(coords)
    .apply(pd.Series)
)

train[['away_lat', 'away_lon']] = (
    train['away_comb']
    .map(coords)
    .apply(pd.Series)
)

train[['stadium_lat', 'stadium_lon']] = (
    train['stadium_comb']
    .map(coords)
    .apply(pd.Series)
)

In [172]:
train.head()

,date,home_team,away_team,tournament,city,country,neutral,year,month,day,...,away_capital,home_comb,away_comb,stadium_comb,home_lat,home_lon,away_lat,away_lon,stadium_lat,stadium_lon
0,1993-06-15,Ecuador,Venezuela,COPA,Quito,Ecuador,False,1993,6,15,...,Caracas,"Quito, Ecuador","Caracas, Venezuela","Quito, Ecuador",-0.220164,-78.512327,10.506093,-66.914601,-0.220164,-78.512327
1,1993-06-16,Colombia,Mexico,COPA,Machala,Ecuador,True,1993,6,16,...,Mexico City,"Bogotá, Colombia","Mexico City, Mexico","Machala, Ecuador",4.653382,-74.083633,19.320772,-99.151468,-3.314894,-79.951280
2,1993-06-16,Uruguay,United States,COPA,Ambato,Ecuador,True,1993,6,16,...,Washington D.C.,"Montevideo, Uruguay","Washington D.C., United States","Ambato, Ecuador",-34.905892,-56.191310,38.895098,-77.036385,-1.242241,-78.628759
3,1993-06-17,Argentina,Bolivia,COPA,Guayaquil,Ecuador,True,1993,6,17,...,Sucre,"Buenos Aires, Argentina","Sucre, Bolivia","Guayaquil, Ecuador",-34.609558,-58.388790,-19.047725,-65.259431,-2.190057,-79.886867
4,1993-06-18,Paraguay,Chile,COPA,Cuenca,Ecuador,True,1993,6,18,...,Santiago,"Asunción, Paraguay","Santiago, Chile","Cuenca, Ecuador",-25.280046,-57.634381,-33.437700,-70.651067,-2.897407,-79.004173


In [173]:
def haversine(lat1, lon1, lat2, lon2):
    # Convert degrees to radians
    lat1, lon1, lat2, lon2 = map(
        np.radians, [lat1, lon1, lat2, lon2]
    )

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    )

    c = 2 * np.arcsin(np.sqrt(a))

    # Earth's radius in kilometers
    R = 6371.0

    return R * c

train["home_stadium_distance_km"] = haversine(
    train["home_lat"],
    train["home_lon"],
    train["stadium_lat"],
    train["stadium_lon"],
)

train["away_stadium_distance_km"] = haversine(
    train["away_lat"],
    train["away_lon"],
    train["stadium_lat"],
    train["stadium_lon"],
)

## Ratings

In [174]:
import asyncio
import pandas as pd
from playwright.async_api import async_playwright
import nest_asyncio
nest_asyncio.apply()


async def make_session():
    playwright = await async_playwright().start()
    browser = await playwright.chromium.launch(headless=True)
    context = await browser.new_context(
        user_agent=(
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/124.0.0.0 Safari/537.36"
        ),
        extra_http_headers={
            "Accept-Language": "en-US,en;q=0.9",
            "Referer": "https://www.sofascore.com/",
        }
    )
    page = await context.new_page()
    print("Warming up browser session...")
    await page.goto("https://www.sofascore.com/", wait_until="domcontentloaded")
    await asyncio.sleep(3)
    return playwright, browser, page


async def search_team_id(page, team_name: str) -> int | None:
    url = f"https://www.sofascore.com/api/v1/search/all?q={team_name.replace(' ', '%20')}&page=0"
    try:
        response = await page.goto(url)
        if response.status != 200:
            print(f"    HTTP {response.status}")
            return None

        data = await response.json()
        results = data.get("results", [])

        # First pass: national teams explicitly flagged
        for r in results:
            if r.get("type") != "team":
                continue
            entity = r["entity"]
            if entity.get("sport", {}).get("slug") != "football":
                continue
            if entity.get("national") is True:
                if team_name.lower() in entity.get("name", "").lower():
                    return entity["id"]

        # Second pass: first football team result
        for r in results:
            if r.get("type") != "team":
                continue
            entity = r["entity"]
            if entity.get("sport", {}).get("slug") == "football":
                return entity["id"]

        return None
    except Exception as e:
        print(f"    Error: {e}")
        return None


async def build_team_id_mapping_async(train: pd.DataFrame) -> dict:
    all_teams = pd.concat([train["home_team"], train["away_team"]]).dropna().unique()
    all_teams = sorted(set(str(t).strip() for t in all_teams))
    print(f"Found {len(all_teams)} unique teams. Looking up IDs...\n")

    playwright, browser, page = await make_session()
    team_id_map = {}
    failed = []

    try:
        for team in all_teams:
            print(f"  Searching: {team} ...", end=" ", flush=True)
            team_id = await search_team_id(page, team)
            if team_id:
                print(f"✓ ID = {team_id}")
                team_id_map[team] = team_id
            else:
                print("✗ Not found")
                failed.append(team)
            await asyncio.sleep(1.5)
    finally:
        await browser.close()
        await playwright.stop()

    with open("team_ids.json", "w") as f:
        json.dump(team_id_map, f, indent=2, ensure_ascii=False)

    print(f"\n✅ Done! {len(team_id_map)}/{len(all_teams)} teams mapped.")
    if failed:
        print(f"⚠️  Not found ({len(failed)}): {failed}")
    return team_id_map


# team_id_map = await build_team_id_mapping_async(train)
# team_id_map['Bosnia and Herzegovina'] = 4479
# team_id_map['Ivory Coast'] = 4768
# team_id_map['Republic of Ireland'] = 4693
# team_id_map['Turkey'] = 4700



In [175]:
with open("team_ids.json", "r") as f:
    team_id_map = json.load(f)

In [ ]:
"""
fetch_team_ratings.py
---------------------
For each row in train, looks up home_team and away_team IDs from team_ids.json,
then fetches the average starter rating from their last 3 matches before the row's date.

Output columns added:
    home_fix_1, home_fix_2, home_fix_3  (most recent → oldest)
    away_fix_1, away_fix_2, away_fix_3

Requirements:
    pip install playwright nest_asyncio
    python -m playwright install chromium  (or .venv/bin/python -m playwright install chromium)
"""

import asyncio
import json
import math
import pandas as pd
import nest_asyncio
from datetime import datetime
import datetime as dt  # rename to avoid conflict with playwright/other imports
from playwright.async_api import async_playwright

nest_asyncio.apply()

# ── Load team ID mapping ───────────────────────────────────────────────────────

with open("team_ids.json", "r") as f:
    TEAM_IDS = json.load(f)

# ── Browser session ────────────────────────────────────────────────────────────

async def make_session():
    playwright = await async_playwright().start()
    browser = await playwright.chromium.launch(headless=True)
    context = await browser.new_context(
        user_agent=(
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/124.0.0.0 Safari/537.36"
        ),
        extra_http_headers={
            "Accept-Language": "en-US,en;q=0.9",
            "Referer": "https://www.sofascore.com/",
        }
    )
    page = await context.new_page()
    print("Warming up browser session...")
    await page.goto("https://www.sofascore.com/", wait_until="domcontentloaded")
    await asyncio.sleep(3)
    return playwright, browser, page


async def api_get(page, url: str):
    """Fetch a SofaScore API URL and return parsed JSON, or None on failure."""
    try:
        response = await page.goto(url)
        if response.status == 200:
            return await response.json()
        return None
    except Exception as e:
        print(f"    ⚠ Request failed ({url}): {e}")
        return None


# ── Core logic ─────────────────────────────────────────────────────────────────

async def get_starter_avg_rating(page, event_id: int, team_id: int, event: dict) -> float | None:
    """Get average starter rating for team_id in event_id."""
    url = f"https://www.sofascore.com/api/v1/event/{event_id}/lineups"
    data = await api_get(page, url)
    if not data:
        return None

    await asyncio.sleep(0.5)

    # Determine which side the team is on using the event dict
    if event["homeTeam"]["id"] == team_id:
        side = "home"
    elif event["awayTeam"]["id"] == team_id:
        side = "away"
    else:
        return None

    players = data.get(side, {}).get("players", [])
    ratings = []
    for p in players:
        if p.get("substitute", True):  # skip substitutes
            continue
        rating = p.get("statistics", {}).get("rating")
        if rating is not None:
            try:
                r = float(rating)
                if not math.isnan(r):
                    ratings.append(r)
            except:
                pass

    return round(sum(ratings) / len(ratings), 3) if ratings else None

async def get_last_3_ratings(page, team_id: int, before_timestamp: int) -> list:
    ratings = []
    page_num = 0

    while len(ratings) < 3:
        url = f"https://www.sofascore.com/api/v1/team/{team_id}/events/last/{page_num}"
        data = await api_get(page, url)
        await asyncio.sleep(1.0)

        if not data:
            break

        events = data.get("events", [])
        if not events:
            break

        for event in events:
            if event.get("startTimestamp", 0) >= before_timestamp:
                continue
            if event.get("status", {}).get("type") != "finished":
                continue

            event_id = event["id"]
            rating = await get_starter_avg_rating(page, event_id, team_id, event)  # pass event
            ratings.append(rating)

            if len(ratings) == 3:
                break

        page_num += 1
        if page_num > 10:
            break

    while len(ratings) < 3:
        ratings.append(None)

    return ratings

async def append_ratings_async(train: pd.DataFrame) -> pd.DataFrame:
    """
    Iterates over train rows, fetches ratings, and returns train with new columns:
    home_fix_1, home_fix_2, home_fix_3, away_fix_1, away_fix_2, away_fix_3
    """
    # Initialize output columns
    for col in ["home_fix_1", "home_fix_2", "home_fix_3",
                "away_fix_1", "away_fix_2", "away_fix_3"]:
        train[col] = None

    playwright, browser, page = await make_session()

    # Cache: {(team_id, date_timestamp): [r1, r2, r3]} — avoids re-fetching same team+date
    cache = {}
    try:
        with open("ratings_cache.json", "r") as f:
            raw = json.load(f)
            cache = {tuple(int(x) for x in k.split("_")): v for k, v in raw.items()}
        print(f"✅ Loaded {len(cache)} cached entries")
    except FileNotFoundError:
        pass

    try:
        for idx, row in train.iterrows():
            date = pd.to_datetime(row["date"])
            timestamp = int(dt.datetime(date.year, date.month, date.day).timestamp())

            home_team = str(row["home_team"]).strip()
            away_team = str(row["away_team"]).strip()

            home_id = TEAM_IDS.get(home_team)
            away_id = TEAM_IDS.get(away_team)

            print(f"\nRow {idx}: {home_team} vs {away_team} ({date.date()})")

            # ── Home team ratings ──
            if home_id is None:
                print(f"  ⚠ No ID for '{home_team}', skipping.")
                home_ratings = [None, None, None]
            else:
                cache_key = (home_id, timestamp)
                if cache_key in cache:
                    home_ratings = cache[cache_key]
                    print(f"  Home: {home_team} (cached) → {home_ratings}")
                else:
                    home_ratings = await get_last_3_ratings(page, home_id, timestamp)
                    cache[cache_key] = home_ratings

                    with open("ratings_cache.json", "w") as f:
                        json.dump({f"{k[0]}_{k[1]}": v for k, v in cache.items()}, f)
                    print(f"  Home: {home_team} → {home_ratings}")

            # ── Away team ratings ──
            if away_id is None:
                print(f"  ⚠ No ID for '{away_team}', skipping.")
                away_ratings = [None, None, None]
            else:
                cache_key = (away_id, timestamp)
                if cache_key in cache:
                    away_ratings = cache[cache_key]
                    print(f"  Away: {away_team} (cached) → {away_ratings}")
                else:
                    away_ratings = await get_last_3_ratings(page, away_id, timestamp)
                    cache[cache_key] = away_ratings

                    with open("ratings_cache.json", "w") as f:
                        json.dump({f"{k[0]}_{k[1]}": v for k, v in cache.items()}, f)
                    print(f"  Away: {away_team} → {away_ratings}")

            # ── Write to dataframe ──
            train.at[idx, "home_fix_1"] = home_ratings[0]
            train.at[idx, "home_fix_2"] = home_ratings[1]
            train.at[idx, "home_fix_3"] = home_ratings[2]
            train.at[idx, "away_fix_1"] = away_ratings[0]
            train.at[idx, "away_fix_2"] = away_ratings[1]
            train.at[idx, "away_fix_3"] = away_ratings[2]

    finally:
        await browser.close()
        await playwright.stop()

    return train



async def run(train: pd.DataFrame) -> pd.DataFrame:
    """Call this with: train = await run(train)"""
    return await append_ratings_async(train)



In [190]:
ratings_df = await run(train)

Warming up browser session...

Row 0: Ecuador vs Venezuela (1993-06-15)
  Home: Ecuador → [None, None, None]
  Away: Venezuela → [None, None, None]

Row 1: Colombia vs Mexico (1993-06-16)
  Home: Colombia → [None, None, None]
  Away: Mexico → [None, None, None]

Row 2: Uruguay vs United States (1993-06-16)


Future exception was never retrieved
future: <Future finished exception=Exception('Connection closed while reading from the driver')>
Exception: Connection closed while reading from the driver
Future exception was never retrieved
future: <Future finished exception=Exception('Connection closed while reading from the driver')>
Exception: Connection closed while reading from the driver


  Home: Uruguay → [7.191, 6.955, 7.5]
  Away: United States → [None, None, None]

Row 3: Argentina vs Bolivia (1993-06-17)
  Home: Argentina → [7.0, 7.709, 7.6]
  Away: Bolivia → [None, None, None]

Row 4: Paraguay vs Chile (1993-06-18)
  Home: Paraguay → [None, None, None]
  Away: Chile → [None, None, None]

Row 5: Brazil vs Peru (1993-06-18)
  Home: Brazil → [7.373, 7.473, 6.7]
  Away: Peru → [6.964, 6.727, None]

Row 6: Ecuador vs United States (1993-06-19)
  Home: Ecuador → [None, None, None]
  Away: United States → [None, None, None]

Row 7: Uruguay vs Venezuela (1993-06-19)
  Home: Uruguay → [7.191, 6.955, 7.5]
  Away: Venezuela → [None, None, None]

Row 8: Argentina vs Mexico (1993-06-20)
  Home: Argentina → [7.0, 7.709, 7.6]
  Away: Mexico → [None, None, None]

Row 9: Colombia vs Bolivia (1993-06-20)
  Home: Colombia → [None, None, None]
  Away: Bolivia → [None, None, None]

Row 10: Brazil vs Chile (1993-06-21)
  Home: Brazil → [7.373, 7.473, 6.7]
  Away: Chile → [None, None, N

Future exception was never retrieved
future: <Future finished exception=Exception('Connection closed while reading from the driver')>
Exception: Connection closed while reading from the driver


  Home: Chile → [None, None, None]
  Away: Peru → [6.964, 6.727, None]

Row 17: Brazil vs Paraguay (1993-06-24)
  Home: Brazil → [7.373, 7.473, 6.7]
  Away: Paraguay → [None, None, None]

Row 18: Ecuador vs Paraguay (1993-06-26)
  Home: Ecuador → [None, None, None]
  Away: Paraguay → [None, None, None]

Row 19: Colombia vs Uruguay (1993-06-26)
  Home: Colombia → [None, None, None]
  Away: Uruguay → [7.191, 6.955, 7.5]

Row 20: Brazil vs Argentina (1993-06-27)
  Home: Brazil → [7.373, 7.473, 6.7]
  Away: Argentina → [7.0, 7.709, 7.6]

Row 21: Peru vs Mexico (1993-06-27)
  Home: Peru → [6.964, 6.727, None]
  Away: Mexico → [None, None, None]

Row 22: Ecuador vs Mexico (1993-06-30)
  Home: Ecuador → [None, None, None]
  Away: Mexico → [None, None, None]

Row 23: Colombia vs Argentina (1993-07-01)
  Home: Colombia → [None, None, None]
  Away: Argentina → [7.0, 7.709, 7.6]

Row 24: Ecuador vs Colombia (1993-07-03)
  Home: Ecuador → [None, None, None]
  Away: Colombia → [None, None, None]

R

Future exception was never retrieved
future: <Future finished exception=Exception('Connection closed while reading from the driver')>
Exception: Connection closed while reading from the driver


  Away: Gabon → [None, None, None]

Row 27: Tunisia vs Mali (1994-03-26)
  Home: Tunisia → [7.545, 7.109, 7.209]
  Away: Mali → [None, None, None]

Row 28: Ghana vs Guinea (1994-03-27)
  Home: Ghana → [None, None, None]
  Away: Guinea → [None, None, None]

Row 29: Ivory Coast vs Sierra Leone (1994-03-27)
  Home: Ivory Coast → [None, None, None]
  Away: Sierra Leone → [None, None, None]

Row 31: Egypt vs Gabon (1994-03-28)
  Home: Egypt → [6.973, 6.818, None]
  Away: Gabon → [None, None, None]

Row 32: Senegal vs Guinea (1994-03-29)
  Home: Senegal → [None, None, None]
  Away: Guinea → [None, None, None]

Row 33: Zambia vs Sierra Leone (1994-03-29)
  Home: Zambia → [None, None, None]
  Away: Sierra Leone → [None, None, None]

Row 34: Nigeria vs Egypt (1994-03-30)
  Home: Nigeria → [None, None, None]
  Away: Egypt → [6.973, 6.818, None]

Row 36: Zambia vs Ivory Coast (1994-03-31)
  Home: Zambia → [None, None, None]
  Away: Ivory Coast → [None, None, None]

Row 37: Ghana vs Senegal (1994-

Future exception was never retrieved
future: <Future finished exception=Exception('Connection closed while reading from the driver')>
Exception: Connection closed while reading from the driver
Future exception was never retrieved
future: <Future finished exception=Exception('Connection closed while reading from the driver')>
Exception: Connection closed while reading from the driver


  Home: Nigeria → [None, None, None]
  Away: Ivory Coast → [None, None, None]

Row 44: Ivory Coast vs Mali (1994-04-10)
  Home: Ivory Coast → [None, None, None]
  Away: Mali → [None, None, None]

Row 45: Nigeria vs Zambia (1994-04-10)
  Home: Nigeria → [None, None, None]
  Away: Zambia → [None, None, None]

Row 46: Germany vs Bolivia (1994-06-17)
  Home: Germany → [None, None, None]
  Away: Bolivia → [None, None, None]

Row 47: Spain vs South Korea (1994-06-17)
  Home: Spain → [7.145, 6.891, 6.727]
  Away: South Korea → [None, None, 6.718]

Row 48: United States vs Switzerland (1994-06-18)
  Home: United States → [None, None, None]
  Away: Switzerland → [None, None, None]

Row 49: Italy vs Republic of Ireland (1994-06-18)
  Home: Italy → [6.991, 7.209, 7.327]
  Away: Republic of Ireland → [7.218, 6.809, 6.718]

Row 50: Colombia vs Romania (1994-06-18)
  Home: Colombia → [None, None, None]
  Away: Romania → [None, None, None]

Row 51: Cameroon vs Sweden (1994-06-19)
  Home: Cameroon → [

Exception: Browser.close: Connection closed while reading from the driver

## Weather

In [106]:
import requests

def get_weather_url(lat, lon, date):
        
    try:
        history = requests.get(
            "https://archive-api.open-meteo.com/v1/archive",
            params={
                    "latitude": lat,
                    "longitude": lon,
                    "start_date": date,
                    "end_date": date,
                    "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum,wind_speed_10m_max",
                    "timezone": "auto"
                }
            ).json()
        return {
            'temperature_max' : history['daily']['temperature_2m_max'][0],
            'temperature_min' : history['daily']['temperature_2m_min'][0],
            'precipitation' : history['daily']['precipitation_sum'][0],
            'wind_speed' : history['daily']['wind_speed_10m_max'][0]
        }
    except:
        raise Exception(f"Error for {lat}, {lon}, {date}")

In [107]:
import json

CACHE_FILE = "weather_cache.json"

# Load cache
try:
    with open(CACHE_FILE) as f:
        weather = json.load(f)
except FileNotFoundError:
    weather = {}

def cache_key(lat, lon, date):
    return f"{lat}_{lon}_{date}"  

def get_weather(lat, lon, date):
    key = cache_key(lat, lon, date)

    if key in weather:
        print(f"Cache hit for {key}")
        return weather[key]  

    for attempt in range(5):
        try:
            time.sleep(1.2)  # respect 600/min limit (~1 req/sec)
            result = get_weather_url(lat, lon, date)

            weather[key] = result
            print("result is back for ", (lon, lat, date))
            with open(CACHE_FILE, "w") as f:
                json.dump(weather, f)

            return result

        except Exception as e:
            wait = 60
            print(f"Error: {e}, waiting {wait}s (attempt {attempt+1}/5)")
            time.sleep(wait)

    print(f"Giving up on ({lat}, {lon}, {date}) after 5 attempts")
    weather[key] = None
    return None

In [ ]:

unique = train[['home_lat', 'home_lon', 'date']].drop_duplicates()

unique['weather'] = unique.apply(
    lambda row: get_weather(row['home_lat'], row['home_lon'], row['date']),
    axis=1
)

weather_df = unique['weather'].apply(pd.Series).add_prefix("home_")
unique = pd.concat([unique, weather_df], axis=1).drop(columns='weather')

train = train.merge(unique, on=['home_lat', 'home_lon', 'date'], how='left')

Error: Error for -0.2201641, -78.5123274, 1993-06-15 00:00:00, waiting 60s (attempt 1/5)


KeyboardInterrupt: 

In [ ]:

unique = train[['away_lat', 'away_lon', 'date']].drop_duplicates()

unique['weather'] = unique.apply(
    lambda row: get_weather(row['away_lat'], row['away_lon'], row['date']),
    axis=1
)

weather_df = unique['weather'].apply(pd.Series).add_prefix("away_")
unique = pd.concat([unique, weather_df], axis=1).drop(columns='weather')

train = train.merge(unique, on=['away_lat', 'away_lon', 'date'], how='left')

In [ ]:

unique = train[['stadium_lat', 'stadium_lon', 'date']].drop_duplicates()

unique['weather'] = unique.apply(
    lambda row: get_weather(row['stadium_lat'], row['stadium_lon'], row['date']),
    axis=1
)

weather_df = unique['weather'].apply(pd.Series).add_prefix("stadium_")
unique = pd.concat([unique, weather_df], axis=1).drop(columns='weather')

train = train.merge(unique, on=['stadium_lat', 'stadium_lon', 'date'], how='left')

In [ ]:
train["home_stadium_temp_max"] = np.abs(train['home_temperature_max'] - train['stadium_temperature_max'])
train["home_stadium_temp_min"] = np.abs(train['home_temperature_min'] - train['stadium_temperature_min'])
train["home_stadium_temp_avg"] = (train['home_stadium_temp_max'] + train['home_stadium_temp_min']) / 2

train["away_stadium_temp_max"] = np.abs(train['away_temperature_max'] - train['stadium_temperature_max'])
train["away_stadium_temp_min"] = np.abs(train['away_temperature_min'] - train['stadium_temperature_min'])
train["away_stadium_temp_avg"] = (train['away_stadium_temp_max'] + train['away_stadium_temp_min']) / 2